# MyAI 100M Train — Leg 1 (GPU)
Resume from aloksingh9182/myai-100m-base (current checkpoint). One logical cosine run to 122,000 steps (~2B tokens on the shared 92.2M-token pack); this leg stops at step 11,000. Exact resume: RNG/optimizer/scaler/data_pos.

In [ ]:
!git clone https://github.com/aloksingh9667/llm.git myai 2>&1 | tail -1
%cd /kaggle/working/myai
!pip install -q -r requirements/train.txt 2>&1 | tail -1


In [ ]:
!ls /kaggle/input
!find /kaggle/input -name 'final.pt' 2>/dev/null


In [ ]:
%%bash
cd /kaggle/working/myai
CKPT=$(find /kaggle/input -name 'final.pt' 2>/dev/null | head -1)
test -n "${CKPT}" || { echo NO_CKPT; exit 1; }
DATA=$(dirname "$(find /kaggle/input -name 'train.pt' 2>/dev/null | head -1)")
test -f "${DATA}/manifest.json" || { echo NO_DATA; exit 1; }
echo "resume-from: ${CKPT}  data-dir: ${DATA}"
python scripts/train.py --model-config configs/myai-100m.yaml \
    --data-dir "${DATA}" \
    --max-steps 122000 --stop-step 11000 \
    --batch-size 4 --accum 8 --amp --no-shuffle \
    --eval-every 500 --ckpt-every 2500 \
    --resume "${CKPT}" \
    --out-dir /kaggle/working/ckpts-leg1 \
    --report /kaggle/working/train-leg1.json


In [ ]:
!test -f /kaggle/working/ckpts-leg1/final.pt && echo LEG1_OK
!python -c "import json; r=json.load(open('/kaggle/working/train-leg1.json')); [print(k, r.get(k)) for k in ('steps','first_loss','final_train_loss','val_loss','tokens_seen')]"